# Phase 8: Informer Transformer Modelling

This notebook trains the Informer — a Transformer-based model with:
- **Multi-head self-attention** encoder (2 layers)
- **Convolutional distilling** between encoder layers (halves sequence length, the Informer's distinctive component)
- **Cross-attention decoder** (1 layer)
- Generative-style decoder: last `label_len=48` history steps as start token, 28 future steps predicted at once

**Architecture parameters** follow the plan: `seq_len=180`, `label_len=48`, `pred_len=28`, `d_model=512`, `n_heads=8`.

Reference: *Informer: Beyond Efficient Transformer for Long Sequence Time-Series Forecasting* (Zhou et al., 2021).

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

DEVICE = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f'Device: {DEVICE}')

PREDS_DIR  = Path('../outputs/predictions')
MODELS_DIR = Path('../outputs/models')
FIGS_DIR   = Path('../outputs/figures')
for d in [PREDS_DIR, MODELS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Device: mps


## 8.1 Load Data

In [2]:
# ── Memory-efficient data loading ─────────────────────────────────────────
# train.parquet has 56 M rows x 67 cols (~34 GB uncompressed).
# We load only required columns and optionally sample series.
# Set N_SERIES=None to use all 30,490 series (~8 GB for these cols).
N_SERIES  = 5000   # set to None for full dataset
LOAD_COLS = ['id', 'date', 'sales', 'cat_id', 'dept_id', 'store_id', 'state_id', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_month_start', 'is_month_end', 'is_event', 'is_snap', 'is_cultural', 'is_national', 'is_religious', 'is_sporting', 'sell_price', 'price_vs_category_mean', 'is_price_reduced', 'price_change_1w', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28']

def _load_split(path, cols=LOAD_COLS, sample_ids=None):
    df = pd.read_parquet(path, columns=cols)
    df['id'] = df['id'].astype('category')
    if sample_ids is not None:
        df = df[df['id'].isin(sample_ids)].copy()
        df['id'] = df['id'].cat.remove_unused_categories()
    return df

# Stratified sample balanced across cat_id x store_id
_meta = pd.read_parquet('../data/features/train.parquet',
                        columns=['id','cat_id','store_id']).drop_duplicates('id')
if N_SERIES and N_SERIES < len(_meta):
    _meta = _meta.groupby(['cat_id','store_id'], group_keys=False).apply(
        lambda g: g.sample(max(1, round(N_SERIES * len(g) / len(_meta))),
                           random_state=42)
    ).head(N_SERIES)
    SAMPLE_IDS = set(_meta['id'].astype(str))
else:
    SAMPLE_IDS = None

train_df = _load_split('../data/features/train.parquet', sample_ids=SAMPLE_IDS)
val_df   = _load_split('../data/features/val.parquet',   sample_ids=SAMPLE_IDS)
test_df  = _load_split('../data/features/test.parquet',  sample_ids=SAMPLE_IDS)

print(f'N_SERIES={N_SERIES}  (None = all 30490)')
print(f'Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}')
print(f'RAM (train): {train_df.memory_usage(deep=True).sum()/1e6:.0f} MB')


N_SERIES=5000  (None = all 30490)
Train: (9285000, 31)  Val: (140000, 31)  Test: (140000, 31)
RAM (train): 892 MB


## 8.2 Feature Selection and Normalisation

In [3]:
# All features fed to encoder (and the history portion of the decoder)
CANDIDATE_ALL_FEATURES = [
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend',
    'is_month_start', 'is_month_end',
    'is_event', 'is_snap', 'is_cultural', 'is_national', 'is_religious', 'is_sporting',
    'sell_price', 'price_vs_category_mean', 'is_price_reduced', 'price_change_1w',
    'lag_28', 'lag_35', 'lag_42', 'lag_56',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28',
    'cat_id', 'dept_id', 'store_id', 'state_id',
]
FEATURE_COLS = [c for c in CANDIDATE_ALL_FEATURES if c in train_df.columns]
N_FEATURES   = len(FEATURE_COLS)
print(f'Using {N_FEATURES} features: {FEATURE_COLS}')

# Per-series mean-sales scale for TARGET normalisation
scale_dict = train_df.groupby('id')['sales'].mean().to_dict()
scale_dict = {k: max(float(v), 1.0) for k, v in scale_dict.items()}

# StandardScaler for features
scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS].fillna(0).values.astype(np.float32))

def scale_df(df):
    df = df.copy()
    df[FEATURE_COLS] = scaler.transform(
        df[FEATURE_COLS].fillna(0).values.astype(np.float32)
    ).astype(np.float32)
    return df

train_sc = scale_df(train_df)
val_sc   = scale_df(val_df)
test_sc  = scale_df(test_df)
print('Feature scaling done.')

Using 28 features: ['dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_month_start', 'is_month_end', 'is_event', 'is_snap', 'is_cultural', 'is_national', 'is_religious', 'is_sporting', 'sell_price', 'price_vs_category_mean', 'is_price_reduced', 'price_change_1w', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28', 'cat_id', 'dept_id', 'store_id', 'state_id']
Feature scaling done.


## 8.3 Informer Dataset

Each sample contains:
- `enc_x` (seq_len=180 steps × features) — full encoder history
- `dec_x` (label_len+pred_len=76 steps × features) — last 48 history + 28 future;  
  unknown-future features (lags, rolling) are zeroed for the last 28 rows
- `target` (28,) — normalised sales for the forecast horizon

In [8]:
from src.models.informer_model import InformerConfig, InformerModel, InformerDataset

SEQ_LEN   = 180
LABEL_LEN = 48
PRED_LEN  = 28
STRIDE    = 28

cfg = InformerConfig()
cfg.enc_in    = N_FEATURES
cfg.dec_in    = N_FEATURES
cfg.seq_len   = SEQ_LEN
cfg.label_len = LABEL_LEN
cfg.pred_len  = PRED_LEN
cfg.d_model   = 256   # reduced from 512 for CPU/MPS friendliness
cfg.n_heads   = 8
cfg.e_layers  = 2
cfg.d_layers  = 1
cfg.dropout   = 0.05
cfg.lr        = 1e-4
cfg.max_epochs = 30
cfg.patience   = 5

print('InformerConfig:', vars(cfg))

InformerConfig: {'enc_in': 28, 'dec_in': 28, 'seq_len': 180, 'label_len': 48, 'pred_len': 28, 'd_model': 256, 'n_heads': 8, 'e_layers': 2, 'd_layers': 1, 'dropout': 0.05, 'lr': 0.0001, 'max_epochs': 30, 'patience': 5}


In [9]:
full_dataset = InformerDataset(
    train_sc, FEATURE_COLS, scale_dict,
    seq_len=SEQ_LEN, label_len=LABEL_LEN, pred_len=PRED_LEN, stride=STRIDE
)
print(f'Total training samples: {len(full_dataset):,}')

val_size   = int(0.10 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds_monitor = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds_monitor, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Verify batch shapes
enc_x, dec_x, target = next(iter(train_loader))
print(f'enc_x: {enc_x.shape}  dec_x: {dec_x.shape}  target: {target.shape}')

Total training samples: 295,000
enc_x: torch.Size([64, 180, 28])  dec_x: torch.Size([64, 76, 28])  target: torch.Size([64, 28])


## 8.4 Instantiate and Train Informer

In [10]:
informer = InformerModel(cfg).to(DEVICE)
n_params = sum(p.numel() for p in informer.parameters() if p.requires_grad)
print(f'InformerModel trainable parameters: {n_params:,}')

InformerModel trainable parameters: 3,041,793


In [11]:
optimizer = torch.optim.Adam(informer.parameters(), lr=cfg.lr)
criterion = nn.MSELoss()

best_val_loss = float('inf')
epochs_no_improve = 0
train_losses_hist = []
val_losses_hist   = []
best_state        = None

for epoch in range(cfg.max_epochs):
    informer.train()
    batch_losses = []
    for enc_x, dec_x, targets in train_loader:
        enc_x, dec_x, targets = enc_x.to(DEVICE), dec_x.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        pred = informer(enc_x, dec_x).squeeze(-1)
        loss = criterion(pred, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(informer.parameters(), 1.0)
        optimizer.step()
        batch_losses.append(loss.item())

    informer.eval()
    val_batch_losses = []
    with torch.no_grad():
        for enc_x, dec_x, targets in val_loader:
            enc_x, dec_x, targets = enc_x.to(DEVICE), dec_x.to(DEVICE), targets.to(DEVICE)
            pred = informer(enc_x, dec_x).squeeze(-1)
            val_batch_losses.append(criterion(pred, targets).item())

    tl = float(np.mean(batch_losses))
    vl = float(np.mean(val_batch_losses))
    train_losses_hist.append(tl)
    val_losses_hist.append(vl)
    print(f'Epoch {epoch+1:3d}/{cfg.max_epochs} | train={tl:.4f} | val={vl:.4f}')

    if vl < best_val_loss:
        best_val_loss = vl
        best_state = {k: v.clone() for k, v in informer.state_dict().items()}
        torch.save(best_state, MODELS_DIR / 'informer_best.pt')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= cfg.patience:
            print(f'Early stopping at epoch {epoch+1}.')
            break

informer.load_state_dict(torch.load(MODELS_DIR / 'informer_best.pt', map_location=DEVICE))
print(f'\nBest val loss: {best_val_loss:.4f}')

Epoch   1/30 | train=0.8997 | val=0.8571
Epoch   2/30 | train=0.8859 | val=0.8535
Epoch   3/30 | train=0.8814 | val=0.8554
Epoch   4/30 | train=0.8781 | val=0.8491
Epoch   5/30 | train=0.8748 | val=0.8519
Epoch   6/30 | train=0.8716 | val=0.8443
Epoch   7/30 | train=0.8679 | val=0.8466
Epoch   8/30 | train=0.8642 | val=0.8458
Epoch   9/30 | train=0.8599 | val=0.8424
Epoch  10/30 | train=0.8554 | val=0.8381
Epoch  11/30 | train=0.8509 | val=0.8378
Epoch  12/30 | train=0.8458 | val=0.8362
Epoch  13/30 | train=0.8406 | val=0.8437
Epoch  14/30 | train=0.8361 | val=0.8341
Epoch  15/30 | train=0.8310 | val=0.8354
Epoch  16/30 | train=0.8259 | val=0.8327
Epoch  17/30 | train=0.8206 | val=0.8342
Epoch  18/30 | train=0.8161 | val=0.8319
Epoch  19/30 | train=0.8118 | val=0.8352
Epoch  20/30 | train=0.8056 | val=0.8279
Epoch  21/30 | train=0.8015 | val=0.8303
Epoch  22/30 | train=0.7963 | val=0.8322
Epoch  23/30 | train=0.7925 | val=0.8279
Epoch  24/30 | train=0.7872 | val=0.8356
Epoch  25/30 | t

In [12]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses_hist, label='Train MSE Loss')
ax.plot(val_losses_hist,   label='Val MSE Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss (normalised units)')
ax.set_title('Informer Training and Validation Loss')
ax.legend(); plt.tight_layout()
plt.savefig(FIGS_DIR / 'informer_loss_curve.png', dpi=150); plt.show()
print('Saved informer_loss_curve.png')

Saved informer_loss_curve.png


## 8.5 Inference

For each series:
- Encoder input: last `seq_len=180` rows from history (scaled)
- Decoder input: last `label_len=48` history rows + 48 zeros for unknown future features
- Decoder known features (calendar, price) are filled from the target period

In [13]:
KNOWN_COLS = [
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend',
    'is_event', 'is_snap', 'is_cultural', 'is_national', 'is_religious', 'is_sporting',
    'sell_price', 'price_vs_category_mean', 'is_price_reduced',
]
KNOWN_COLS = [c for c in KNOWN_COLS if c in FEATURE_COLS]
unknown_mask = np.array([c not in KNOWN_COLS for c in FEATURE_COLS], dtype=bool)

def informer_predict(model, history_sc, future_sc, feature_cols,
                     scale_dict, device,
                     seq_len=180, label_len=48, pred_len=28):
    """Generate 28-day predictions for every series in future_sc."""
    model.eval()
    records = []
    for sid in future_sc['id'].unique():
        hist = history_sc[history_sc['id'] == sid].sort_values('date').tail(seq_len)
        fut  = future_sc[future_sc['id'] == sid].sort_values('date').head(pred_len)
        if len(hist) < seq_len:
            continue

        enc_arr = hist[feature_cols].values.astype(np.float32)  # (180, F)

        # Decoder: last label_len history rows + future known features
        dec_hist_arr = hist[feature_cols].values[-label_len:].copy()         # (48, F)
        dec_fut_arr  = fut[feature_cols].values[:pred_len].copy()            # (28, F)
        dec_fut_arr[:, unknown_mask] = 0.0                                   # zero unknowns
        dec_arr = np.concatenate([dec_hist_arr, dec_fut_arr], axis=0)        # (76, F)

        enc_t = torch.tensor(enc_arr, dtype=torch.float32).unsqueeze(0).to(device)
        dec_t = torch.tensor(dec_arr, dtype=torch.float32).unsqueeze(0).to(device)

        scale = scale_dict.get(sid, 1.0)
        with torch.no_grad():
            pred_norm = model(enc_t, dec_t).squeeze(0).squeeze(-1).cpu().numpy()

        pred_sales = np.clip(pred_norm * scale, 0.0, None)
        for d, p, a in zip(fut['date'].values,
                            pred_sales[:len(fut)],
                            fut['sales'].values):
            records.append({'id': sid, 'date': d,
                            'predicted': float(p), 'actual': float(a)})
    return pd.DataFrame(records)


val_preds = informer_predict(
    informer, train_sc, val_sc, FEATURE_COLS, scale_dict, DEVICE,
    seq_len=SEQ_LEN, label_len=LABEL_LEN, pred_len=PRED_LEN
)
val_preds.to_csv(PREDS_DIR / 'informer_val_preds.csv', index=False)
print(f'Val predictions saved: {len(val_preds):,} rows')

Val predictions saved: 140,000 rows


In [14]:
trainval_sc = scale_df(pd.concat([train_df, val_df], ignore_index=True))

test_preds = informer_predict(
    informer, trainval_sc, test_sc, FEATURE_COLS, scale_dict, DEVICE,
    seq_len=SEQ_LEN, label_len=LABEL_LEN, pred_len=PRED_LEN
)
test_preds.to_csv(PREDS_DIR / 'informer_test_preds.csv', index=False)
print(f'Test predictions saved: {len(test_preds):,} rows')

Test predictions saved: 140,000 rows


## 8.6 Evaluation Metrics

In [15]:
from src.evaluation.metrics import compute_all_metrics

for split_name, preds_df in [('Validation', val_preds), ('Test', test_preds)]:
    m = compute_all_metrics(preds_df['actual'].values, preds_df['predicted'].values)
    print(f'\n=== {split_name} Metrics ===')
    for k, v in m.items():
        print(f'  {k}: {v:.4f}')


=== Validation Metrics ===
  MAE: 1.1953
  RMSE: 2.7497
  MAPE: 3653693244.1446
  sMAPE: 145.9651

=== Test Metrics ===
  MAE: 1.2056
  RMSE: 2.7442
  MAPE: 3621498945.6792
  sMAPE: 141.8671


In [16]:
print('\n=== Validation Metrics by Category ===')
combined = val_preds.merge(
    val_df[['id', 'date', 'cat_id']].drop_duplicates(), on=['id', 'date'], how='left'
)
for cat, grp in combined.groupby('cat_id'):
    m = compute_all_metrics(grp['actual'].values, grp['predicted'].values)
    print(f'  cat={cat} | MAE={m["MAE"]:.3f} | RMSE={m["RMSE"]:.3f} | sMAPE={m["sMAPE"]:.2f}%')


=== Validation Metrics by Category ===
  cat=0 | MAE=1.671 | RMSE=3.648 | sMAPE=132.39%
  cat=1 | MAE=0.629 | RMSE=1.202 | sMAPE=167.02%
  cat=2 | MAE=0.846 | RMSE=1.718 | sMAPE=153.25%
